# A2.4 · The NHI governance gap

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.3 · Shadow Autonomy](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, SPIRE |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Non-human identities outnumber human ones in a typical estate by somewhere
between 10:1 and 50:1. They are governed by roughly none of the same process:
no joiner-mover-leaver, no manager attestation, no periodic recertification,
frequently no owner.

The gap is not primarily a policy gap. Policies for this exist and are easy to
write. The gap is that **nobody can enumerate them**, and you cannot govern,
tier, revoke or recertify a list you do not have.

Agents make this acute for three reasons:

- They are created **programmatically**, often per task, so the population grows
  without anyone filing a request.
- They are frequently created by *other* agents, so the requester is not a
  person who can be asked.
- They **outlive their purpose**. A retired agent whose identity still exists is
  a standing credential with no owner and no expiry — the single most common
  finding in a first NHI review.

So the first control is not a policy. It is an inventory, built from telemetry
rather than from a survey nobody answers.

## 2 · Demo — build the inventory from telemetry

The registry below is what a survey produces. The auth log is what is actually happening. The interesting rows are the ones in one and not the other.

In [ ]:
import time
from dataclasses import dataclass, field

now = time.time(); DAY = 86400

# what the CMDB / spreadsheet says exists
REGISTERED = {
    "ci-builder":    {"owner": "platform", "created": now - 900*DAY},
    "deploy-bot":    {"owner": "platform", "created": now - 700*DAY},
    "triage-agent":  {"owner": "appsec",   "created": now - 120*DAY},
    "backup-runner": {"owner": "",         "created": now - 1400*DAY},
}
# what the identity provider actually saw authenticate in the last 90 days
AUTH_LOG = {
    "ci-builder":        now - 0.2*DAY,
    "deploy-bot":        now - 1*DAY,
    "triage-agent":      now - 0.1*DAY,
    "svc-legacy-etl":    now - 3*DAY,      # not in the registry at all
    "agent-worker-7f3c": now - 0.05*DAY,   # spawned by another agent
    "agent-worker-a91b": now - 0.05*DAY,
    # backup-runner: absent — has not authenticated in 90 days
}

registered, seen = set(REGISTERED), set(AUTH_LOG)
print(f"{'identity':22s}{'in registry':13s}{'seen in logs':14s}{'owner':10s}status")
print("-" * 74)
for ident in sorted(registered | seen):
    in_reg, in_log = ident in registered, ident in seen
    owner = REGISTERED.get(ident, {}).get("owner") or "—"
    if in_reg and in_log:   status = "governed" if owner != "—" else "NO OWNER"
    elif in_log:            status = "SHADOW — unregistered but active"
    else:                   status = "ORPHAN — registered, never authenticates"
    print(f"{ident:22s}{str(in_reg):13s}{str(in_log):14s}{owner:10s}{status}")

## 3 · Where it breaks

Three of the six active identities were never registered, and one registered identity has not authenticated in 90 days. Both directions are findings, and they fail differently:

- **Shadow** identities have no owner, so no one can answer "should this exist?" during an incident.
- **Orphans** are standing credentials for a purpose that ended. Nobody will notice them being used, because nobody is watching something they believe is retired.

In [ ]:
def gaps(ident):
    out = []
    reg = REGISTERED.get(ident)
    if reg is None:
        out.append("unregistered: in use but never requested or approved")
    elif not reg["owner"]:
        out.append("no named owner — nobody can accept the risk or recertify it")
    last = AUTH_LOG.get(ident)
    if last is None:
        age = (now - reg["created"]) / DAY if reg else 0
        out.append(f"no authentication in 90d — decommissioning never finished "
                   f"(identity is {age:.0f}d old)")
    if ident.startswith("agent-worker-"):
        out.append("created by another agent — no human requester exists")
    return out

for ident in sorted(registered | seen):
    g = gaps(ident)
    if g:
        print(f"{ident}")
        for x in g:
            print(f"   ⚠ {x}")

## 4 · The control — per-identity revocation, proven

An inventory is only useful if you can act on a single row. The test that separates an *identity* from a *shared password* is: can you revoke exactly one of these without breaking the others?

In [ ]:
class Registry:
    def __init__(self):
        self.revoked, self.issued = set(), []
    def issue(self, actor, scopes):
        t = {"actor": actor, "scopes": set(scopes), "id": len(self.issued)}
        self.issued.append(t); return t
    def revoke(self, actor):
        self.revoked.add(actor)
        return sum(1 for t in self.issued if t["actor"] == actor)
    def valid(self, t):
        return t["actor"] not in self.revoked

reg = Registry()
toks = {a: reg.issue(a, {"repo:read"}) for a in
        ("ci-builder", "deploy-bot", "agent-worker-7f3c", "agent-worker-a91b")}

print("revoking one dynamically-spawned worker:")
n = reg.revoke("agent-worker-7f3c")
print(f"   invalidated {n} token(s)\n")
for a, t in toks.items():
    print(f"   {a:22s} valid={reg.valid(t)}")

print("\nNow the shared-secret world, for contrast:")
SHARED = "one API key used by all four"
print(f"   revoking the shared key stops: {list(toks)}")
print("   → which is why, in practice, nobody ever revokes it.")

In [ ]:
# Verify: the inventory has to be reproducible, not a one-off spreadsheet.
def build_inventory(registry, auth_log, stale_days=90):
    rows = []
    for ident in sorted(set(registry) | set(auth_log)):
        reg = registry.get(ident)
        last = auth_log.get(ident)
        rows.append({
            "identity": ident,
            "owner": (reg or {}).get("owner") or None,
            "registered": reg is not None,
            "active": last is not None,
            "stale": last is None,
            "auto_spawned": ident.startswith("agent-worker-"),
        })
    return rows

inv = build_inventory(REGISTERED, AUTH_LOG)
unowned = [r["identity"] for r in inv if not r["owner"]]
shadow  = [r["identity"] for r in inv if not r["registered"]]
stale   = [r["identity"] for r in inv if r["stale"]]
print(f"identities: {len(inv)}   unowned: {len(unowned)}   "
      f"shadow: {len(shadow)}   stale: {len(stale)}")
print(f"   unowned {unowned}\n   shadow  {shadow}\n   stale   {stale}")
assert shadow and stale, "a first inventory always finds both"

## What you just proved

Seven identities appear across the registry and the auth log: three governed, one with no owner, three shadow (including two agent-spawned workers) and one orphan. Revoking `agent-worker-7f3c` invalidates only its own token. The inventory reports 4 unowned, 3 shadow and 1 stale.

## Your turn

Run the real query: every non-human identity in one production account, joined against 90 days of authentication events. Count the rows with no owner. That number, not a policy document, is your NHI governance gap.

---

**Next → [A2.5 · Delegation that survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*